# Stochastic inventory control by policy iteration

A port of `stochinventorycontroPI.mlx`.

The inventory problem of Unit 2, solved a third way. Unit 2 used stochastic
dynamic programming; `code/unit10/` learned it by Q-learning; here it is
solved by **policy iteration**, which alternates two steps:

- **policy evaluation** -- fix the policy and compute its cost. With the
  policy fixed there is no minimisation left, so this is a backward sweep
  (here) or a linear system (in the infinite-horizon case).
- **policy improvement** -- one greedy sweep against the costs just
  computed. This is exactly a rollout step with the current policy as the
  base policy.

The loop stops when the improvement step returns the policy it was given,
which must happen in finitely many steps because there are finitely many
policies.

This notebook reproduces the slide sequence of the unit: the initial policy
"buy nothing", then $\mu^1,\dots,\mu^4$, then $\mu^5=\mu^4$ and stop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (14, 4)

from policy_iteration import policy_iteration

In [ ]:
p, citem, ctr, c1_ = 10, 5, 0.5, 0.2
d = [2, 2, 1, 1]            # demand; at stage 3 (index 2) it is random
h, N = 4, 4                 # stages, maximum stock

DM = [[[] for _ in range(N+1)] for _ in range(h)]   # next state
CM = [[[] for _ in range(N+1)] for _ in range(h)]   # cost
PM = [[[] for _ in range(N+1)] for _ in range(h)]   # probability

for k in range(h):
    for i in range(N+1):                            # stock level
        if k != 2:                                  # deterministic demand
            for j in range(0, N-i+1):               # order quantity
                c2 = citem*j + (ctr if j > 0 else 0); c1 = i*c1_
                DM[k][i].append([max(i+j-d[k], 0)])
                CM[k][i].append([-(min(d[k], j+i)*p - (c1+c2))])
                PM[k][i].append([1.0])
        else:                                       # demand 1 or 0, each w.p. 1/2
            d3 = 0
            for j in range(0, max(N+d[k]-i, N+d3-i) + 1):
                nxt, cst, prb = [], [], []
                for dd_, pr_ in ((d[k], 0.5), (d3, 0.5)):
                    if j <= N + dd_ - i:
                        c2 = citem*j + (ctr if j > 0 else 0); c1 = i*c1_
                        nxt.append(max(i+j-dd_, 0))
                        cst.append(-(min(dd_, j+i)*p - (c1+c2)))
                        prb.append(pr_)
                DM[k][i].append(nxt); CM[k][i].append(cst); PM[k][i].append(prb)

gh = [-(x*(citem - c1_)) for x in range(N+1)]       # salvage value of leftover stock
mdp = {"DM": DM, "CM": CM, "PM": PM, "gh": gh}
print(f"{h} stages, {N+1} stock levels, terminal costs {gh}")

In [ ]:
mu, J, history = policy_iteration(mdp)
print(f"converged after {len(history)} policy iterations\n")
for ell, (mu_l, J_l) in enumerate(history):
    print(f"  mu^{ell}:  J(stock 0, stage 0) = {J_l[0][0]:8.2f}")

Those are the numbers on the slides: $0$ for the do-nothing policy
$\mu^0$, then $-25.30$, $-25.70$, $-25.75$ and finally $-25.95$ -- the
sequence of lattice diagrams in the lecture, one per improvement step.

In [ ]:
J_table = np.array([[J[k][i] for k in range(h+1)] for i in range(N+1)])
print("optimal cost, rows = stock 0..4, columns = stage 0..4:")
print(np.round(J_table, 2))

mu_table = np.array([[mu[k][i] for k in range(h)] for i in range(N+1)])
print("\noptimal order quantity, rows = stock 0..4, columns = stage 0..3:")
print(mu_table)

The stage-0 column, $-25.95,\,-30.75,\,-36.05,\,-40.65,\,-45.45$, is the
top row of the lattice on the lecture slide, and it agrees to the last digit
with two independent calculations elsewhere in the course: the stochastic DP
of Unit 2, and the exact-DP reference inside
`code/unit10/A_10_StochInventoryControlQLearning.ipynb`.

Three routes to the same five numbers is the real content of this notebook.
Policy iteration gets there in five evaluations rather than one backward
sweep, which is more work here -- its advantage appears in the
infinite-horizon case, where DP would need many value-iteration sweeps while
policy evaluation is a single linear solve.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(N+1):
    ax.plot([J_l[0][i] for _, J_l in history], "o-", label=f"stock {i}")
ax.set_xlabel("policy iteration $\\ell$")
ax.set_ylabel("$J_{\\mu^\\ell}$ at stage 0")
ax.grid(True); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()